In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, ParameterGrid
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle
import shutil
import os
from google.colab import drive

In [ ]:
drive.mount('/content/drive')
print("Google Drive successfully mounted.")

In [ ]:
# Automatically create the 'machine_learning_models' folder if it doesn't exist
DRIVE_MODEL_PATH = '/content/drive/MyDrive/machine_learning_models'

# Create the folder if it doesn't exist
os.makedirs(DRIVE_MODEL_PATH, exist_ok=True)
print(f"Target folder '{DRIVE_MODEL_PATH}' checked/created.")

# Filenames to save
MODEL_FILENAME = 'random_forest_churn_model.pkl'
SCALER_FILENAME = 'standard_scaler.pkl'

In [ ]:
url = "https://raw.githubusercontent.com/uras-alkaya/AI-Projects/refs/heads/main/churn_features_final.csv"
df = pd.read_csv(url)

X = df.drop("churn", axis=1)
y = df["churn"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

print("Original Distribution:", Counter(y_train))
print("Balanced Distribution:", Counter(y_train_res))

In [ ]:
# Parameter grid for RandomForestClassifier
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 7],
}

# Manual GridSearch
grid = ParameterGrid(param_grid)
best_score = 0
best_params = None

print("GridSearch for RandomForestClassifier has started...\n")

for params in tqdm(grid, desc="RandomForest GridSearch Progress"):
    model = RandomForestClassifier(**params, random_state=42)
    model.fit(X_train_scaled, y_train_res)
    y_pred = model.predict(X_test_scaled)

    score = f1_score(y_test, y_pred)
    print(f"Trying parameters: {params} | F1 Score: {score:.4f}")

    if score > best_score:
        best_score = score
        best_params = params

print(f"\nBest Params: {best_params}")
print(f"Best Score: {best_score:.4f}")

In [ ]:
# After GridSearch, train the final model using the best parameters
best_params = {'max_depth': 10, 'min_samples_split': 7, 'n_estimators': 100}
final_model = RandomForestClassifier(**best_params, random_state=42)
final_model.fit(X_train_scaled, y_train_res)

In [ ]:
# First, save the model temporarily in Colab directory
temp_model_path = os.path.join('/content', MODEL_FILENAME)
with open(temp_model_path, 'wb') as file:
    pickle.dump(final_model, file)
print(f"Model '{MODEL_FILENAME}' temporarily saved.")

temp_scaler_path = os.path.join('/content', SCALER_FILENAME)
with open(temp_scaler_path, 'wb') as file:
    pickle.dump(scaler, file)
print(f"Scaler '{SCALER_FILENAME}' temporarily saved.")

# Copy temporary files to Google Drive
shutil.copy(temp_model_path, os.path.join(DRIVE_MODEL_PATH, MODEL_FILENAME))
shutil.copy(temp_scaler_path, os.path.join(DRIVE_MODEL_PATH, SCALER_FILENAME))

print(f"\nModel and scaler successfully copied to '{DRIVE_MODEL_PATH}' folder.")

In [ ]:
# Full paths of model and scaler to be loaded from Google Drive
LOAD_MODEL_PATH = os.path.join(DRIVE_MODEL_PATH, MODEL_FILENAME)
LOAD_SCALER_PATH = os.path.join(DRIVE_MODEL_PATH, SCALER_FILENAME)

loaded_model = None
loaded_scaler = None

# Load Model
try:
    with open(LOAD_MODEL_PATH, 'rb') as file:
        loaded_model = pickle.load(file)
    print(f"Model '{LOAD_MODEL_PATH}' successfully loaded.")
except FileNotFoundError:
    print(f"Error: Model file '{LOAD_MODEL_PATH}' not found. Please check the path or ensure the model is trained and saved first.")
except Exception as e:
    print(f"An error occurred while loading the model: {e}")

# Load Scaler
try:
    with open(LOAD_SCALER_PATH, 'rb') as file:
        loaded_scaler = pickle.load(file)
    print(f"Scaler '{LOAD_SCALER_PATH}' successfully loaded.")
except FileNotFoundError:
    print(f"Error: Scaler file '{LOAD_SCALER_PATH}' not found. Please check the path or ensure the scaler is trained and saved first.")
except Exception as e:
    print(f"An error occurred while loading the scaler: {e}")

In [ ]:
# Make predictions with the final model
y_pred_final = loaded_model.predict(X_test_scaled)
y_pred_proba_final = loaded_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate final model
print("F1 Score:", f1_score(y_test, y_pred_final))
print("Accuracy:", accuracy_score(y_test, y_pred_final))
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba_final))
print("\nClassification Report:\n", classification_report(y_test, y_pred_final))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_final))

In [ ]:
# Plot ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_final)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc_score(y_test, y_pred_proba_final):.2f})')
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve (Loaded Model)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
# Confusion Matrix Visualization
print("\n--- Confusion Matrix Visualization ---\n")
cm = confusion_matrix(y_test, y_pred_final)
plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Negative (0)', 'Predicted Positive (1)'],
            yticklabels=['Actual Negative (0)', 'Actual Positive (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# --- Prediction Results Table ---
results_df = pd.DataFrame({
    'True_Churn': y_test.values,
    'Predicted_Churn': y_pred_final,
    'Predicted_Probability_Churn': y_pred_proba_final
})

print("\n--- Prediction Results Table ---\n")
print(results_df)

In [ ]:
# Find incorrect predictions
incorrect_predictions = results_df[results_df['True_Churn'] != results_df['Predicted_Churn']]

print("\n--- Rows Where Model Made Mistakes ---")
print(incorrect_predictions)

In [ ]:
incorrect_predictions = results_df[results_df['True_Churn'] != results_df['Predicted_Churn']]

false_positives = incorrect_predictions[incorrect_predictions['True_Churn'] == 0]
false_negatives = incorrect_predictions[incorrect_predictions['True_Churn'] == 1]

print(f"\nTotal Number of Incorrect Predictions: {len(incorrect_predictions)}")
print(f"False Positives (FP): {len(false_positives)}")
print(f"False Negatives (FN): {len(false_negatives)}")

In [ ]:
# Feature Importance Scores
feature_importances = loaded_model.feature_importances_
features = X.columns

# Convert feature importances to DataFrame
importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

print("\nFeature Importance Scores (RandomForestClassfier):\n")
print(importance_df)

In [ ]:
# Visualize Feature Importances
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importances')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Parameter grid for LogisticRegression
param_grid_lr = {
    'C': [0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

grid_lr = ParameterGrid(param_grid_lr)
best_score_lr = 0
best_params_lr = None

print("GridSearch for Logistic Regression has started...\n")

for params in tqdm(grid_lr, desc="LR GridSearch Progress"):
    model_lr = LogisticRegression(**params, random_state=42)
    model_lr.fit(X_train_scaled, y_train_res)
    y_pred_lr = model_lr.predict(X_test_scaled)

    score_lr = f1_score(y_test, y_pred_lr)

    print(f"Trying parameters: {params} | F1 Score: {score_lr:.4f}")

    if score_lr > best_score_lr:
        best_score_lr = score_lr
        best_params_lr = params

print(f"\nBest Params for Logistic Regression: {best_params_lr}")
print(f"Best Score for Logistic Regression: {best_score_lr:.4f}")

In [ ]:
best_params_lr = {'C': 100, 'penalty': 'l2', 'solver': 'liblinear'}
final_model_lr = LogisticRegression(**best_params_lr, random_state=42)
final_model_lr.fit(X_train_scaled, y_train_res)

In [ ]:
# File name for Logistic Regression model
LR_MODEL_FILENAME = 'logistic_regression_churn_model.pkl'

# Save temporarily to Colab's directory
temp_lr_model_path = os.path.join('/content', LR_MODEL_FILENAME)
with open(temp_lr_model_path, 'wb') as file:
    pickle.dump(final_model_lr, file)
print(f"Logistic Regression model '{LR_MODEL_FILENAME}' saved temporarily.")

# Copy the temporary file to Google Drive
shutil.copy(temp_lr_model_path, os.path.join(DRIVE_MODEL_PATH, LR_MODEL_FILENAME))
print(f"\nLogistic Regression model successfully copied to '{DRIVE_MODEL_PATH}' folder.")

In [ ]:
LOAD_LR_MODEL_PATH = os.path.join(DRIVE_MODEL_PATH, LR_MODEL_FILENAME)

loaded_lr_model = None

# Load the Logistic Regression model
try:
    with open(LOAD_LR_MODEL_PATH, 'rb') as file:
        loaded_lr_model = pickle.load(file)
    print(f"Logistic Regression model loaded successfully from '{LOAD_LR_MODEL_PATH}'.")
except FileNotFoundError:
    print(f"Error: Logistic Regression model file '{LOAD_LR_MODEL_PATH}' not found. Please check the path or train and save the model first.")
except Exception as e:
    print(f"An error occurred while loading the Logistic Regression model: {e}")

In [ ]:
# Make predictions using the final Logistic Regression model
y_pred_lr_final = loaded_lr_model.predict(X_test_scaled)
y_pred_proba_lr_final = loaded_lr_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate the final Logistic Regression model
print("F1 Score (Logistic Regression):", f1_score(y_test, y_pred_lr_final))
print("Accuracy (Logistic Regression):", accuracy_score(y_test, y_pred_lr_final))
print("ROC AUC (Logistic Regression):", roc_auc_score(y_test, y_pred_proba_lr_final))
print("\nClassification Report (Logistic Regression):\n", classification_report(y_test, y_pred_lr_final))
print("\nConfusion Matrix (Logistic Regression):\n", confusion_matrix(y_test, y_pred_lr_final))

In [ ]:
# ROC Curve Plot (Logistic Regression)
fpr_lr, tpr_lr, thresholds_lr = roc_curve(y_test, y_pred_proba_lr_final)
plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, color='green', lw=2, label=f'ROC curve (area = {roc_auc_score(y_test, y_pred_proba_lr_final):.2f})')
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve (Loaded Logistic Regression Model)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
# Confusion Matrix Visualization (Logistic Regression)
print("\n--- Confusion Matrix Visualization (Logistic Regression) ---\n")
cm_lr = confusion_matrix(y_test, y_pred_lr_final)
plt.figure(figsize=(6, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Predicted Negative (0)', 'Predicted Positive (1)'],
            yticklabels=['Actual Negative (0)', 'Actual Positive (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix (Logistic Regression)')
plt.show()

In [ ]:
# --- Prediction Results Table (Logistic Regression) ---
results_df_lr = pd.DataFrame({
    'True_Churn': y_test.values,
    'Predicted_Churn_LR': y_pred_lr_final,
    'Predicted_Probability_Churn_LR': y_pred_proba_lr_final
})

print("\n--- Prediction Results Table (Logistic Regression) ---")
print(results_df_lr)

In [ ]:
# Find incorrect predictions (Logistic Regression)
incorrect_predictions_lr = results_df_lr[results_df_lr['True_Churn'] != results_df_lr['Predicted_Churn_LR']]

print("\n--- Incorrect Predictions by Logistic Regression Model ---")
print(incorrect_predictions_lr)

In [ ]:
false_positives_lr = incorrect_predictions_lr[incorrect_predictions_lr['True_Churn'] == 0]
false_negatives_lr = incorrect_predictions_lr[incorrect_predictions_lr['True_Churn'] == 1]

print(f"\nTotal Number of Incorrect Predictions (Logistic Regression): {len(incorrect_predictions_lr)}")
print(f"False Positives (FP) Count (Logistic Regression): {len(false_positives_lr)}")
print(f"False Negatives (FN) Count (Logistic Regression): {len(false_negatives_lr)}")

In [ ]:
# Get model coefficients
lr_coefficients = loaded_lr_model.coef_[0]
features = X.columns

# Convert feature importances to a DataFrame
importance_df_lr = pd.DataFrame({'Feature': features, 'Importance': abs(lr_coefficients)})
importance_df_lr = importance_df_lr.sort_values(by='Importance', ascending=False)

print("\nFeature Importance Scores (Logistic Regression - Absolute Coefficients):\n")
print(importance_df_lr)

In [ ]:
# Feature Importance Visualization
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df_lr)
plt.title('Feature Importances (Logistic Regression - Absolute Coefficients)')
plt.xlabel('Importance Score (Absolute Coefficient Value)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Parameter grid for XGBoost Classifier
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}

grid_xgb = ParameterGrid(param_grid_xgb)
best_score_xgb = 0
best_params_xgb = None

print("GridSearch for XGBoost has started...\n")

for params in tqdm(grid_xgb, desc="XGB GridSearch Progress"):
    model_xgb = XGBClassifier(**params, random_state=42, eval_metric='logloss')
    model_xgb.fit(X_train_scaled, y_train_res)
    y_pred_xgb = model_xgb.predict(X_test_scaled)

    score_xgb = f1_score(y_test, y_pred_xgb)
    print(f"Trying parameters: {params} | F1 Score: {score_xgb:.4f}")

    if score_xgb > best_score_xgb:
        best_score_xgb = score_xgb
        best_params_xgb = params

print(f"\nBest Params for XGBoost: {best_params_xgb}")
print(f"Best F1 Score for XGBoost: {best_score_xgb:.4f}")

In [ ]:
best_params_xgb = {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}
final_model_xgb = XGBClassifier(**best_params_xgb, random_state=42, eval_metric='logloss')
final_model_xgb.fit(X_train_scaled, y_train_res)

In [ ]:
# Filename for XGBoost Classifier model
XGB_MODEL_FILENAME = 'xgboost_churn_model.pkl'

# Save temporarily to Colab's directory
temp_xgb_model_path = os.path.join('/content', XGB_MODEL_FILENAME)
with open(temp_xgb_model_path, 'wb') as file:
    pickle.dump(final_model_xgb, file)
print(f"XGBoost model '{XGB_MODEL_FILENAME}' saved temporarily.")

# Copy the temporary file to Google Drive
shutil.copy(temp_xgb_model_path, os.path.join(DRIVE_MODEL_PATH, XGB_MODEL_FILENAME))
print(f"\nXGBoost model successfully copied to '{DRIVE_MODEL_PATH}' folder.")

In [ ]:
LOAD_XGB_MODEL_PATH = os.path.join(DRIVE_MODEL_PATH, XGB_MODEL_FILENAME)

loaded_xgb_model = None

# Load the XGBoost model
try:
    with open(LOAD_XGB_MODEL_PATH, 'rb') as file:
        loaded_xgb_model = pickle.load(file)
    print(f"XGBoost model loaded successfully from '{LOAD_XGB_MODEL_PATH}'.")
except FileNotFoundError:
    print(f"Error: XGBoost model file '{LOAD_XGB_MODEL_PATH}' not found. Please check the path or train and save the model first.")
except Exception as e:
    print(f"An error occurred while loading the XGBoost model: {e}")

In [ ]:
# Make predictions using the final XGBoost model
y_pred_xgb_final = loaded_xgb_model.predict(X_test_scaled)
y_pred_proba_xgb_final = loaded_xgb_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate the final XGBoost model
print("F1 Score (XGB):", f1_score(y_test, y_pred_xgb_final))
print("Accuracy (XGB):", accuracy_score(y_test, y_pred_xgb_final))
print("ROC AUC (XGB):", roc_auc_score(y_test, y_pred_proba_xgb_final))
print("\nClassification Report (XGB):\n", classification_report(y_test, y_pred_xgb_final))
print("\nConfusion Matrix (XGB):\n", confusion_matrix(y_test, y_pred_xgb_final))

In [ ]:
# ROC Curve Plot (XGBoost)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_pred_proba_xgb_final)
plt.figure(figsize=(8, 6))
plt.plot(fpr_xgb, tpr_xgb, color='orange', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_test, y_pred_proba_xgb_final):.2f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curve (XGBoost)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
# --- Confusion Matrix Visualization (XGBoost) ---
print("\n--- Confusion Matrix Visualization (XGBoost) ---\n")
cm_xgb = confusion_matrix(y_test, y_pred_xgb_final)
plt.figure(figsize=(6, 6))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['Predicted Negative (0)', 'Predicted Positive (1)'],
            yticklabels=['Actual Negative (0)', 'Actual Positive (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Confusion Matrix (XGBoost)')
plt.show()

In [ ]:
# --- Prediction Results Table (XGBoost) ---
results_df_xgb = pd.DataFrame({
    'True_Churn': y_test.values,
    'Predicted_Churn_XGB': y_pred_xgb_final,
    'Predicted_Probability_Churn_XGB': y_pred_proba_xgb_final
})

print("\n--- Prediction Results Table (XGBoost) ---")
print(results_df_xgb)

In [ ]:
# Find incorrect predictions (XGBoost)
incorrect_predictions_xgb = results_df_xgb[results_df_xgb['True_Churn'] != results_df_xgb['Predicted_Churn_XGB']]

print("\n--- Incorrect Predictions by XGBoost Model ---")
print(incorrect_predictions_xgb)

In [ ]:
false_positives_xgb = incorrect_predictions_xgb[incorrect_predictions_xgb['True_Churn'] == 0]
false_negatives_xgb = incorrect_predictions_xgb[incorrect_predictions_xgb['True_Churn'] == 1]

print(f"\nTotal Number of Incorrect Predictions (XGBoost): {len(incorrect_predictions_xgb)}")
print(f"False Positives (FP) Count (XGBoost): {len(false_positives_xgb)}")
print(f"False Negatives (FN) Count (XGBoost): {len(false_negatives_xgb)}")

In [ ]:
# Feature Importance Scores (XGBoost)
feature_importances_xgb = loaded_xgb_model.feature_importances_
features = X.columns

# Convert feature importances to DataFrame
importance_df_xgb = pd.DataFrame({'Feature': features, 'Importance': feature_importances_xgb})
importance_df_xgb = importance_df_xgb.sort_values(by='Importance', ascending=False)

print("\nFeature Importance Scores (XGBoost):\n")
print(importance_df_xgb)

In [ ]:
# Feature Importance Visualization
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df_xgb)
plt.title('XGBoost Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Parameter grid for LinearSVC
param_grid_linear_svc = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'loss': ['squared_hinge'],
    'dual': [False],
    'max_iter': [5000, 10000]
}

grid_linear_svc = ParameterGrid(param_grid_linear_svc)
best_score_linear_svc = 0
best_params_linear_svc = None

print("\nLinearSVC GridSearch Started...\n")

for params in tqdm(grid_linear_svc, desc="LinearSVC GridSearch Progress"):
        model_linear_svc = LinearSVC(**params, random_state=42)
        model_linear_svc.fit(X_train_scaled, y_train_res)
        y_pred_linear_svc = model_linear_svc.predict(X_test_scaled)

        score_linear_svc = f1_score(y_test, y_pred_linear_svc)
        print(f"Trying parameters: {params} | F1 Score: {score_linear_svc:.4f}")

        if score_linear_svc > best_score_linear_svc:
            best_score_linear_svc = score_linear_svc
            best_params_linear_svc = params

print(f"\nBest Params for LinearSVC: {best_params_linear_svc}")
print(f"Best F1 Score for LinearSVC: {best_score_linear_svc:.4f}")

In [ ]:
best_params_linear_svc = {'C': 0.1, 'dual': False, 'loss': 'squared_hinge', 'max_iter': 5000, 'penalty': 'l1'}
final_model_linear_svc = LinearSVC(**best_params_linear_svc, random_state=42)
final_model_linear_svc.fit(X_train_scaled, y_train_res)

In [ ]:
# File name for the LinearSVC model
LINEAR_SVC_MODEL_FILENAME = 'linear_svc_churn_model.pkl'

# First save to Colab's temporary directory
temp_linear_svc_model_path = os.path.join('/content', LINEAR_SVC_MODEL_FILENAME)
with open(temp_linear_svc_model_path, 'wb') as file:
    pickle.dump(final_model_linear_svc, file)
print(f"LinearSVC model '{LINEAR_SVC_MODEL_FILENAME}' temporarily saved.")

# Copy the temporary file to Google Drive
shutil.copy(temp_linear_svc_model_path, os.path.join(DRIVE_MODEL_PATH, LINEAR_SVC_MODEL_FILENAME))
print(f"\nLinearSVC model successfully copied to '{DRIVE_MODEL_PATH}' folder.")

In [ ]:
LOAD_LINEAR_SVC_MODEL_PATH = os.path.join(DRIVE_MODEL_PATH, LINEAR_SVC_MODEL_FILENAME)

loaded_linear_svc_model = None

# Load the LinearSVC model
try:
    with open(LOAD_LINEAR_SVC_MODEL_PATH, 'rb') as file:
        loaded_linear_svc_model = pickle.load(file)
    print(f"LinearSVC model loaded successfully from '{LOAD_LINEAR_SVC_MODEL_PATH}'.")
except FileNotFoundError:
    print(f"Error: LinearSVC model file '{LOAD_LINEAR_SVC_MODEL_PATH}' not found. Please check the path or train and save the model first.")
except Exception as e:
    print(f"An error occurred while loading the LinearSVC model: {e}")

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

y_pred_linear_svc_final = loaded_linear_svc_model.predict(X_test_scaled)

calibrated_linear_svc = CalibratedClassifierCV(loaded_linear_svc_model, method='sigmoid', cv='prefit')
calibrated_linear_svc.fit(X_train_scaled, y_train_res)
y_pred_proba_linear_svc_final_calibrated = calibrated_linear_svc.predict_proba(X_test_scaled)[:, 1]

print("F1 Score (LinearSVC):", f1_score(y_test, y_pred_linear_svc_final))
print("Accuracy (LinearSVC):", accuracy_score(y_test, y_pred_linear_svc_final))
print("ROC AUC (LinearSVC):", roc_auc_score(y_test, y_pred_proba_linear_svc_final_calibrated))
print("\nClassification Report (LinearSVC):\n", classification_report(y_test, y_pred_linear_svc_final))
print("\nConfusion Matrix (LinearSVC):\n", confusion_matrix(y_test, y_pred_linear_svc_final))


In [ ]:
# ROC Curve Plot (Calibrated LinearSVC)
fpr_linear_svc_calibrated, tpr_linear_svc_calibrated, _ = roc_curve(y_test, y_pred_proba_linear_svc_final_calibrated)
plt.figure(figsize=(8, 6))
plt.plot(fpr_linear_svc_calibrated, tpr_linear_svc_calibrated, color='purple', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_test, y_pred_proba_linear_svc_final_calibrated):.2f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curve (Calibrated LinearSVC)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
# --- Confusion Matrix Visualization (LinearSVC) ---
print("\n--- Confusion Matrix Visualization (LinearSVC) ---\n")
cm_linear_svc = confusion_matrix(y_test, y_pred_linear_svc_final)
plt.figure(figsize=(6, 6))
sns.heatmap(cm_linear_svc, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Predicted Negative (0)', 'Predicted Positive (1)'],
            yticklabels=['Actual Negative (0)', 'Actual Positive (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (LinearSVC)')
plt.show()

In [ ]:
# --- Prediction Results Table (LinearSVC) ---
results_df_linear_svc = pd.DataFrame({
    'True_Churn': y_test.values,
    'Predicted_Churn_LinearSVC': y_pred_linear_svc_final,
    'Predicted_Probability_Churn_LinearSVC': y_pred_proba_linear_svc_final_calibrated
})

print("\n--- Prediction Results Table (LinearSVC) ---\n")
print(results_df_linear_svc)

In [ ]:
# Find incorrect predictions (LinearSVC)
incorrect_predictions_linear_svc = results_df_linear_svc[results_df_linear_svc['True_Churn'] != results_df_linear_svc['Predicted_Churn_LinearSVC']]

print("\n--- Misclassified Rows by LinearSVC Model ---")
print(incorrect_predictions_linear_svc)

In [ ]:
false_positives_linear_svc = incorrect_predictions_linear_svc[incorrect_predictions_linear_svc['True_Churn'] == 0]
false_negatives_linear_svc = incorrect_predictions_linear_svc[incorrect_predictions_linear_svc['True_Churn'] == 1]

print(f"\nTotal Misclassified Samples (LinearSVC): {len(incorrect_predictions_linear_svc)}")
print(f"False Positives (FP) Count (LinearSVC): {len(false_positives_linear_svc)}")
print(f"False Negatives (FN) Count (LinearSVC): {len(false_negatives_linear_svc)}")

In [ ]:
# Feature Importance Scores and Visualization (LinearSVC)
linear_svc_coefficients = loaded_linear_svc_model.coef_[0]
features = X.columns

importance_df_linear_svc = pd.DataFrame({'Feature': features, 'Importance': abs(linear_svc_coefficients)})
importance_df_linear_svc = importance_df_linear_svc.sort_values(by='Importance', ascending=False)

print("\nFeature Importance Scores (LinearSVC - Absolute Coefficients):\n")
print(importance_df_linear_svc)

In [ ]:
# Feature Importance Visualization
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df_linear_svc)
plt.title('Feature Importances (LinearSVC - Absolute Coefficient Values)')
plt.xlabel('Importance Score (Absolute Coefficient Value)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Parameter grid for KNeighborsClassifier
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid_knn = ParameterGrid(param_grid_knn)
best_score_knn = 0
best_params_knn = None

print("\nKNeighborsClassifier GridSearch Started...\n")

for params in tqdm(grid_knn, desc="KNN GridSearch Progress"):
    model_knn = KNeighborsClassifier(**params)
    model_knn.fit(X_train_scaled, y_train_res)
    y_pred_knn = model_knn.predict(X_test_scaled)

    score_knn = f1_score(y_test, y_pred_knn)
    print(f"Trying parameters: {params} | F1 Score: {score_knn:.4f}")

    if score_knn > best_score_knn:
            best_score_knn = score_knn
            best_params_knn = params

print(f"\nBest Params for KNN: {best_params_knn}")
print(f"Best F1 Score for KNN: {best_score_knn:.4f}")

In [ ]:
best_params_knn = {'metric': 'manhattan', 'n_neighbors': 9, 'weights': 'uniform'}
final_model_knn = KNeighborsClassifier(**best_params_knn)
final_model_knn.fit(X_train_scaled, y_train_res)

In [ ]:
# Filename for KNeighborsClassifier model
KNN_MODEL_FILENAME = 'knn_churn_model.pkl'

# Save temporarily to Colab's directory
temp_knn_model_path = os.path.join('/content', KNN_MODEL_FILENAME)
with open(temp_knn_model_path, 'wb') as file:
    pickle.dump(final_model_knn, file)
print(f"KNN model '{KNN_MODEL_FILENAME}' temporarily saved.")

# Copy the temporary file to Google Drive
shutil.copy(temp_knn_model_path, os.path.join(DRIVE_MODEL_PATH, KNN_MODEL_FILENAME))
print(f"\nKNN model successfully copied to '{DRIVE_MODEL_PATH}' folder.")

In [ ]:
LOAD_KNN_MODEL_PATH = os.path.join(DRIVE_MODEL_PATH, KNN_MODEL_FILENAME)

loaded_knn_model = None

# Load the KNN model
try:
    with open(LOAD_KNN_MODEL_PATH, 'rb') as file:
        loaded_knn_model = pickle.load(file)
    print(f"KNN model loaded successfully from '{LOAD_KNN_MODEL_PATH}'.")
except FileNotFoundError:
    print(f"Error: KNN model file '{LOAD_KNN_MODEL_PATH}' not found. Please check the path or train and save the model first.")
except Exception as e:
    print(f"An error occurred while loading the KNN model: {e}")

In [ ]:
# Make predictions with the final KNN model
y_pred_knn_final = loaded_knn_model.predict(X_test_scaled)
y_pred_proba_knn_final = loaded_knn_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate the final KNN model
print("F1 Score (KNN):", f1_score(y_test, y_pred_knn_final))
print("Accuracy (KNN):", accuracy_score(y_test, y_pred_knn_final))
print("ROC AUC (KNN):", roc_auc_score(y_test, y_pred_proba_knn_final))
print("\nClassification Report (KNN):\n", classification_report(y_test, y_pred_knn_final))
print("\nConfusion Matrix (KNN):\n", confusion_matrix(y_test, y_pred_knn_final))

In [ ]:
# --- ROC Curve Plot (KNN) ---
fpr_knn, tpr_knn, thresholds_knn = roc_curve(y_test, y_pred_proba_knn_final)
plt.figure(figsize=(8, 6))
plt.plot(fpr_knn, tpr_knn, color='Turquoise', lw=2, label=f'ROC curve (area = {roc_auc_score(y_test, y_pred_proba_knn_final):.2f})')
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve (Loaded KNN Model)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
# --- Confusion Matrix Visualization (KNN) ---
print("\n--- Confusion Matrix Visualization (KNN) ---\n")
cm_knn = confusion_matrix(y_test, y_pred_knn_final)
plt.figure(figsize=(6, 6))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Negative (0)', 'Predicted Positive (1)'],
            yticklabels=['Actual Negative (0)', 'Actual Positive (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (KNN)')
plt.show()

In [ ]:
# --- Prediction Results Table (KNN) ---
results_df_knn = pd.DataFrame({
    'True_Churn': y_test.values,
    'Predicted_Churn_KNN': y_pred_knn_final,
    'Predicted_Probability_Churn_KNN': y_pred_proba_knn_final
})

print("\n--- Prediction Results Table (KNN) ---")
print(results_df_knn)

In [ ]:
# Find misclassified predictions (KNN)
incorrect_predictions_knn = results_df_knn[results_df_knn['True_Churn'] != results_df_knn['Predicted_Churn_KNN']]

print("\n--- Misclassified Rows by KNN Model ---")
print(incorrect_predictions_knn)

In [ ]:
false_positives_knn = incorrect_predictions_knn[incorrect_predictions_knn['True_Churn'] == 0]
false_negatives_knn = incorrect_predictions_knn[incorrect_predictions_knn['True_Churn'] == 1]

print(f"\nTotal Misclassified Samples (KNN): {len(incorrect_predictions_knn)}")
print(f"False Positives (FP) Count (KNN): {len(false_positives_knn)}")
print(f"False Negatives (FN) Count (KNN): {len(false_negatives_knn)}")

In [ ]:
from sklearn.inspection import permutation_importance
import numpy as np

subset_size = min(1000, X_test_scaled.shape[0])

indices = np.random.choice(X_test_scaled.shape[0], subset_size, replace=False)
X_test_scaled_subset = X_test_scaled[indices]
y_test_subset = y_test.iloc[indices]

print(f"A subset of {subset_size} test samples is used for permutation importance calculation.")

perm_importance_knn = permutation_importance(
    loaded_knn_model,
    X_test_scaled_subset,
    y_test_subset,
    n_repeats=2,
    random_state=42,
    n_jobs=-1
)

importance_df_knn_perm = pd.DataFrame({
    'Feature': X.columns,
    'Importance': perm_importance_knn.importances_mean,
    'Std_Dev': perm_importance_knn.importances_std
})

# Sort by importance scores
importance_df_knn_perm = importance_df_knn_perm.sort_values(by='Importance', ascending=False)
print(importance_df_knn_perm)

In [ ]:
# Feature Importance Visualization
plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df_knn_perm)
plt.title('Feature Importances (KNeighborsClassifier - Permutation Importance)')
plt.xlabel('Importance Score (Mean Decrease in F1 Score)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# Parameter grid for MLPClassifier
param_grid_mlp = {
    'hidden_layer_sizes': [(50,), (100,)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001],
    'max_iter': [500]
}

grid_mlp = ParameterGrid(param_grid_mlp)
best_score_mlp = 0
best_params_mlp = None

print("\nMLPClassifier GridSearch Started...\n")

for params in tqdm(grid_mlp, desc="MLP GridSearch Progress"):
    model_mlp = MLPClassifier(**params, random_state=42)
    model_mlp.fit(X_train_scaled, y_train_res)
    y_pred_mlp = model_mlp.predict(X_test_scaled)

    score_mlp = f1_score(y_test, y_pred_mlp)
    print(f"Trying parameters: {params} | F1 Score: {score_mlp:.4f}")

    if score_mlp > best_score_mlp:
            best_score_mlp = score_mlp
            best_params_mlp = params

print(f"\nBest Params for MLPClassifier: {best_params_mlp}")
print(f"Best F1 Score for MLPClassifier: {best_score_mlp:.4f}")

In [ ]:
best_params_mlp = {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (50,), 'max_iter': 500, 'solver': 'adam'}
final_model_mlp = MLPClassifier(**best_params_mlp, random_state=42)
final_model_mlp.fit(X_train_scaled, y_train_res)

In [ ]:
# Filename for MLPClassifier model
MLP_MODEL_FILENAME = 'mlp_churn_model.pkl'

# Save temporarily to Colab's directory
temp_mlp_model_path = os.path.join('/content', MLP_MODEL_FILENAME)
with open(temp_mlp_model_path, 'wb') as file:
    pickle.dump(final_model_mlp, file)
print(f"MLPClassifier model '{MLP_MODEL_FILENAME}' temporarily saved.")

# Copy the temporary file to Google Drive
shutil.copy(temp_mlp_model_path, os.path.join(DRIVE_MODEL_PATH, MLP_MODEL_FILENAME))
print(f"\nMLPClassifier model successfully copied to '{DRIVE_MODEL_PATH}' folder.")

In [ ]:
LOAD_MLP_MODEL_PATH = os.path.join(DRIVE_MODEL_PATH, MLP_MODEL_FILENAME)

loaded_mlp_model = None

# Load the MLPClassifier model
try:
    with open(LOAD_MLP_MODEL_PATH, 'rb') as file:
        loaded_mlp_model = pickle.load(file)
    print(f"MLPClassifier model loaded successfully from '{LOAD_MLP_MODEL_PATH}'.")
except FileNotFoundError:
    print(f"Error: MLPClassifier model file '{LOAD_MLP_MODEL_PATH}' not found. Please check the path or train and save the model first.")
except Exception as e:
    print(f"An error occurred while loading the MLPClassifier model: {e}")

In [ ]:
y_pred_mlp_final = loaded_mlp_model.predict(X_test_scaled)
y_pred_proba_mlp_final = loaded_mlp_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate the final MLPClassifier model
print("F1 Score (MLP):", f1_score(y_test, y_pred_mlp_final))
print("Accuracy (MLP):", accuracy_score(y_test, y_pred_mlp_final))
print("ROC AUC (MLP):", roc_auc_score(y_test, y_pred_proba_mlp_final))
print("\nClassification Report (MLP):\n", classification_report(y_test, y_pred_mlp_final))
print("\nConfusion Matrix (MLP):\n", confusion_matrix(y_test, y_pred_mlp_final))

In [ ]:
# --- ROC Curve Plot (MLP) ---
fpr_mlp, tpr_mlp, thresholds_mlp = roc_curve(y_test, y_pred_proba_mlp_final)
plt.figure(figsize=(8, 6))
plt.plot(fpr_mlp, tpr_mlp, color='gray', lw=2, label=f'ROC curve (area = {roc_auc_score(y_test, y_pred_proba_mlp_final):.2f})')
plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve (Loaded MLPClassifier Model)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
# --- Confusion Matrix Visualization (MLP) ---
print("\n--- Confusion Matrix Visualization (MLP) ---\n")
cm_mlp = confusion_matrix(y_test, y_pred_mlp_final)
plt.figure(figsize=(6, 6))
sns.heatmap(cm_mlp, annot=True, fmt='d', cmap='Grays', cbar=False,
            xticklabels=['Predicted Negative (0)', 'Predicted Positive (1)'],
            yticklabels=['Actual Negative (0)', 'Actual Positive (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (MLPClassifier)')
plt.show()

In [ ]:
# --- Prediction Results Table (MLP) ---
results_df_mlp = pd.DataFrame({
    'True_Churn': y_test.values,
    'Predicted_Churn_MLP': y_pred_mlp_final,
    'Predicted_Probability_Churn_MLP': y_pred_proba_mlp_final
})

print("\n--- Prediction Results Table (MLP) ---")
print(results_df_mlp)

In [ ]:
# Find misclassified predictions (MLP)
incorrect_predictions_mlp = results_df_mlp[results_df_mlp['True_Churn'] != results_df_mlp['Predicted_Churn_MLP']]

print("\n--- Misclassified Rows by MLPClassifier Model ---")
print(incorrect_predictions_mlp)

In [ ]:
false_positives_mlp = incorrect_predictions_mlp[incorrect_predictions_mlp['True_Churn'] == 0]
false_negatives_mlp = incorrect_predictions_mlp[incorrect_predictions_mlp['True_Churn'] == 1]

print(f"\nTotal Misclassified Samples (MLP): {len(incorrect_predictions_mlp)}")
print(f"False Positives (FP) Count (MLP): {len(false_positives_mlp)}")
print(f"False Negatives (FN) Count (MLP): {len(false_negatives_mlp)}")

In [ ]:
import shap

# Sample 100 points from training data as background for SHAP
background_data = shap.utils.sample(X_train_scaled, 100)

# Sample up to 500 points from test set to speed up SHAP calculation
sample_size_for_shap = min(2000, X_test_scaled.shape[0])
X_test_scaled_sample = shap.utils.sample(X_test_scaled, sample_size_for_shap, random_state=42)

# Initialize SHAP explainer with MLP's predict_proba and background data
explainer_mlp = shap.KernelExplainer(loaded_mlp_model.predict_proba, background_data)

print(f"Calculating SHAP values on {sample_size_for_shap}")
shap_explanation_mlp = explainer_mlp(X_test_scaled_sample)
print("SHAP values calculated.")

# Get SHAP values for class 1
shap_values_for_class_1 = shap_explanation_mlp.values[:, :, 1]

shap_df_mlp = pd.DataFrame(shap_values_for_class_1, columns=X.columns)
mean_abs_shap_values_mlp = shap_df_mlp.abs().mean().sort_values(ascending=False)

print("\nSHAP Feature Importance Scores (MLPClassifier - Mean |SHAP|):\n")
print(mean_abs_shap_values_mlp)

In [ ]:
# Feature Importance Visualization
importance_df_mlp = pd.DataFrame({
    'Feature': mean_abs_shap_values_mlp.index,
    'Importance': mean_abs_shap_values_mlp.values
})

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df_mlp)
plt.title('Feature Importances (MLPClassifier - Mean Absolute SHAP Values)')
plt.xlabel('Mean Absolute SHAP Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()